# Advanced IR: PyTerrier

Completing these tutorial exercises give you the information and tools you need to complete the projects in this course. They are not just a list of exercises, but code snippets and ideas that you should use in your project.

This tutorial assume you are running on a *nix system. If you have trouble running this on Windows, consider using the WSL (Windows subsystem for Linux), or running this notebook in Google Colab.

### Preparation: Install dependencies

First, we install the libraries we need for this tutorial.

In [1]:
!pip3 install ir_datasets pandas python-terrier

### Step 1: Load the dataset and imports

The IR community maintains a common interface to many IR ranking datasets, used for training and evaluation: https://ir-datasets.com/

It has an accompanying library that can access these datasets in a convenient way.

In [2]:
import ir_datasets

# https://ir-datasets.com/beir.html#beir/scifact/test
dataset = ir_datasets.load("beir/scifact/test")

### Step 2: View first Five Topics

The `dataset.queries_iter()` method creates an iterable over all topics in the dataset.
Each topic has an `query_id`, the string that is submitted to the search engine as query (can be accessed via `default_text`).

In [3]:
for query in list(dataset.queries_iter())[:3]:
    print('\nQuery: ', query.query_id)

    print('\tText:\t\t' + query.default_text())


Query:  1
	Text:		0-dimensional biomaterials show inductive properties.

Query:  3
	Text:		1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.

Query:  5
	Text:		1/2000 in UK have abnormal PrP positivity.


### Step 3: View First Five Relevance Judgments

The `dataset.qrels_iter()` method creates an iterable over all relevance judgments (**qrels for query relevance**) of the dataset.
Each qrel entry consists of an `query_id` pointing to a topic (represented by a query above), a `doc_id` pointing to a document from the collection, and a relevance label indicating if a document is relevant (relevance > 0) or not relevant (relevance is 0) to a query. The iteration field is "historically unused".

E.g., the first line below indicates that document `31715818` is relevant to the query `0-dimensional biomaterials show inductive properties.`.


In [4]:
for qrel in list(dataset.qrels_iter())[:5]:
    # Access via: qrel.query_id, qrel.doc_id, qrel.relevance
    print(qrel)

TrecQrel(query_id='1', doc_id='31715818', relevance=1, iteration='0')
TrecQrel(query_id='3', doc_id='14717500', relevance=1, iteration='0')
TrecQrel(query_id='5', doc_id='13734012', relevance=1, iteration='0')
TrecQrel(query_id='13', doc_id='1606628', relevance=1, iteration='0')
TrecQrel(query_id='36', doc_id='5152028', relevance=1, iteration='0')


### Step 4: Access to documents

The `dataset.docs_store()` method provides random access via the document ID to all documents of a corpus.

For instance, `docs_store.get('31715818')` returns the document with id `31715818` that has the text `"Nanotechnologies are emerging platforms that could be useful in measuring, understanding..."`.

The `dataset.docs_iter()` method creates an iterable over all documents in a corpus (can be suitable to build an index).

In [5]:
print('The dataset has', len(list(dataset.docs_iter())), 'documents.')

The dataset has 5183 documents.


In [6]:
docs_store = dataset.docs_store()

docs_store.get('31715818')

BeirTitleDoc(doc_id='31715818', text='Nanotechnologies are emerging platforms that could be useful in measuring, understanding, and manipulating stem cells. Examples include magnetic nanoparticles and quantum dots for stem cell labeling and in vivo tracking; nanoparticles, carbon nanotubes, and polyplexes for the intracellular delivery of genes/oligonucleotides and protein/peptides; and engineered nanometer-scale scaffolds for stem cell differentiation and transplantation. This review examines the use of nanotechnologies for stem cell tracking, differentiation, and transplantation. We further discuss their utility and the potential concerns regarding their cytotoxicity.', title='New opportunities: the use of nanotechnologies to manipulate and track stem cells.')

### Step 5: Create some Descriptive Statistics for the Relevance Judgments

Next, we want to create a [Pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) that shows the proportion of relevant documents per topic.
You can imagine a DataFrame as a table.

First, we show how to use the pandas DataFrame API, second, it is on you to create this table using the real data.

In [7]:
import pandas as pd

df = pd.DataFrame([
    {'query_id': 'test-1', 'query': 'some test query 1', 'proportion_relevant': 0.3},
    {'query_id': 'test-2', 'query': 'some test query 2', 'proportion_relevant': 0.4},
    {'query_id': 'test-3', 'query': 'some test query 3', 'proportion_relevant': 0.2},
])

df.sort_values('proportion_relevant', ascending=False)

,query_id,query,proportion_relevant
1,test-2,some test query 2,0.4
0,test-1,some test query 1,0.3
2,test-3,some test query 3,0.2


E.g., in the hypothetical example above, the query `test-2` has the highest proportion of relevant documents.

Next, please create a pandas DataFrame `df` that reports the proportion of relevant documents per topic on the real data, using `dataset.queries_iter()` and `dataset.qrels_iter()`.

In [8]:
# TODO: create df using dataset.queries_iter() and dataset.qrels_iter():

total_docs = len(list(dataset.docs_iter()))
dict_list = []
for query in list(dataset.queries_iter()):
    relevant_docs = [qrel for qrel in dataset.qrels_iter() if qrel.query_id == query.query_id and qrel.relevance > 0]
    prop_rel = len(relevant_docs) / total_docs
    dict_list.append({
        'query_id': query.query_id,
        'query': query.default_text(),
        'proportion_relevant': prop_rel,
    })
    
df2 = pd.DataFrame(dict_list)
df2.sort_values('proportion_relevant', ascending=False)

,query_id,query,proportion_relevant
191,873,Obesity is determined solely by environmental ...,0.000965
29,133,Assembly of invadopodia is triggered by focal ...,0.000965
209,971,Primary cervical cancer screening with HPV det...,0.000772
38,179,Birth-weight is positively associated with bre...,0.000772
295,1379,Women with a higher birth weight are more like...,0.000772
...,...,...,...
111,532,Hyperfibrinogenemia decreases rates of femorop...,0.000193
112,533,Hyperfibrinogenemia increases rates of femorop...,0.000193
113,535,Hypertension is frequently observed in type 1 ...,0.000193
114,536,Hypocretin neurones induce panicprone state in...,0.000193


### Step 6: Find Difficult Topics

Identify the query with the lowest proportion of relevant documents

In [9]:
# TODO Fill this cell (e.g., via `sort_values` on the `df` from above)
lowest_rel_query = df2.sort_values('proportion_relevant', ascending=True).iloc[0]
print(lowest_rel_query)

query_id                                                               1
query                  0-dimensional biomaterials show inductive prop...
proportion_relevant                                             0.000193
Name: 0, dtype: object


### Step 7: Find Easy Topics

Identify the query with the highest proportion of relevant documents

In [10]:
# TODO Fill this cell (e.g., via `sort_values` on the `df` from above)
highest_rel_query = df2.sort_values('proportion_relevant', ascending=False).iloc[0]
print(highest_rel_query)

query_id                                                             873
query                  Obesity is determined solely by environmental ...
proportion_relevant                                             0.000965
Name: 191, dtype: object


### Step 8: Indexing documents in pyterrier   

PyTerrier is a Python framework for performing information retrieval experiments. It is built on top of terrier (http://terrier.org/) which is a highly flexible, efficient, and effective open source search engine, readily deployable on large-scale collections of documents. Terrier implements state-of-the-art indexing and retrieval functionalities, and provides an ideal platform for the rapid development and evaluation of large-scale retrieval applications. 

We will use PyTerrier a lot in this course, but if you're interested in other tools, pyserini (https://github.com/castorini/pyserini), which is built on top of Lucene (the search engine underlying things like Elasticsearch) can also be used to perform experiments (but focused on different goals).

Much of PyTerrier's view of the world is wrapped up in Pandas dataframes. Let's consider some textual documents in a dataframe.

In [11]:
import pyterrier as pt

# lets not truncate output too much
pd.set_option('display.max_colwidth', 150)

docs_df = pd.DataFrame([
        ["d1", "this is the first document of many documents"],
        ["d2", "this is another document"],
        ["d3", "the topic of this document is unknown"]
    ], columns=["docno", "text"])

docs_df

,docno,text
0,d1,this is the first document of many documents
1,d2,this is another document
2,d3,the topic of this document is unknown


Before any search engine can estimate which documents are most likely to be relevant for a given query, it must index the documents.

In the following cell, we index the dataframe's documents. The index, with all its data structures, is written into a directory called index_3docs.


In [12]:
indexer = pt.DFIndexer("./index_3docs", overwrite=True)
index_ref = indexer.index(docs_df["text"], docs_df["docno"])
index_ref.toString()

/var/folders/ct/242_3ld56f1c5w52hw4j71w00000gn/T/ipykernel_33539/1583266151.py:1: DeprecationWarning: Call to deprecated class DFIndexer. (use pt.terrier.IterDictIndexer().index(dataframe.to_dict(orient='records')) instead) -- Deprecated since version 0.11.0.
  indexer = pt.DFIndexer("./index_3docs", overwrite=True)
Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]



'./index_3docs/data.properties'

An `IndexRef`
 is essentially a string saying where an index is stored. Indeed, we can look in the `index_3docs` directory and see that it has created various small files: 

In [13]:
!ls -lh index_3docs/

total 80
-rw-r--r--@ 1 kavyagnihotri  staff     3B Oct 20 13:35 data.direct.bf
-rw-r--r--@ 1 kavyagnihotri  staff    51B Oct 20 13:35 data.document.fsarrayfile
-rw-r--r--@ 1 kavyagnihotri  staff     4B Oct 20 13:35 data.inverted.bf
-rw-r--r--@ 1 kavyagnihotri  staff   344B Oct 20 13:35 data.lexicon.fsomapfile
-rw-r--r--@ 1 kavyagnihotri  staff   249B Oct 20 13:35 data.lexicon.fsomaphash
-rw-r--r--@ 1 kavyagnihotri  staff    33B Oct 20 13:35 data.meta-0.fsomapfile
-rw-r--r--@ 1 kavyagnihotri  staff    24B Oct 20 13:35 data.meta.idx
-rw-r--r--@ 1 kavyagnihotri  staff    48B Oct 20 13:35 data.meta.zdata
-rw-r--r--@ 1 kavyagnihotri  staff   4.1K Oct 20 13:35 data.properties


With an `IndexRef`, we can load it to an actual index. The method `pt.IndexFactory.of()` is the relevant factory. 

In [14]:
index = pt.IndexFactory.of(index_ref)

#lets see what type index is.
type(index)

jnius.reflect.org.terrier.structures.Index

Ok, so this object refers to Terrier's [`Index`](http://terrier.org/docs/current/javadoc/org/terrier/structures/Index.html) type. Check the linked Javadoc – you will see that this Java object has methods such as:
 - `getCollectionStatistics()`
 - `getInvertedIndex()`
 - `getLexicon()`

Let's see what is returned by the `CollectionStatistics()` method:

In [15]:
print(index.getCollectionStatistics().toString())

Number of documents: 3
Number of terms: 4
Number of postings: 6
Number of fields: 0
Number of tokens: 7
Field names: []
Positions:   false



Ok, that seems fair – we have 3 documents. But why only 4 terms? 
Let's check the [`Lexicon`](http://terrier.org/docs/current/javadoc/org/terrier/structures/Lexicon.html), which is our vocabulary. Fortunately, the `Lexicon` can be iterated easily from Python:

In [16]:
for kv in index.getLexicon():
  print("%s (%s) : %s (%s)" % (kv.getKey(), type(kv.getKey()), kv.getValue().toString(), type(kv.getValue()) ) )

document (<class 'str'>) : term0 Nt=3 TF=4 maxTF=2 @{0 0 0} (<class 'jnius.reflect.org.terrier.structures.LexiconEntry'>)
first (<class 'str'>) : term1 Nt=1 TF=1 maxTF=1 @{0 0 7} (<class 'jnius.reflect.org.terrier.structures.LexiconEntry'>)
topic (<class 'str'>) : term2 Nt=1 TF=1 maxTF=1 @{0 1 1} (<class 'jnius.reflect.org.terrier.structures.LexiconEntry'>)
unknown (<class 'str'>) : term3 Nt=1 TF=1 maxTF=1 @{0 1 5} (<class 'jnius.reflect.org.terrier.structures.LexiconEntry'>)


Here, iterating over the `Lexicon` returns a pair of `String ` term and a [`LexiconEntry`](http://terrier.org/docs/current/javadoc/org/terrier/structures/LexiconEntry.html) object – which itself is an [`EntryStatistics`](http://terrier.org/docs/current/javadoc/org/terrier/structures/EntryStatistics.html) – and contains information including the statistics of that term.


So what did we find? Here are some observations:
 - we only have 4 unique terms, as stopwords were removed;
 - we have one term for `"document"`, even though `"documents"` occurred in document "`d1`". 
 
Both these observations make sense, as indeed Terrier removes standard stopwords and applies Porter's stemmer by default.

Further:
 - `Nt` is the number of unique documents that each term occurs in – this is useful for calculating IDF.
 - `TF` is the total number of occurrences – some weighting models use this instead of Nt.
 - The numbers in the `@{}` are a pointer – they tell Terrier where the postings are for that term in the inverted index data structure.

Finally, we can also use the square bracket notation to lookup terms in Terrier's lexicon:


In [17]:
index.getLexicon()["document"].toString()

'term0 Nt=3 TF=4 maxTF=2 @{0 0 0}'

Let's now think about the inverted index. The inverted index tells us in which *documents* each term occurs in. The `LexiconEntry` is the pointer that tell us where to find the postings for that term in the inverted index.

In [18]:
pointer = index.getLexicon()["document"]
for posting in index.getInvertedIndex().getPostings(pointer):
    print(posting.toString() + " doclen=%d" % posting.getDocumentLength())

ID(0) TF(2) doclen=3
ID(1) TF(1) doclen=1
ID(2) TF(1) doclen=3


Ok, so we can see that `"document"` occurs once in each of the three documents. 

NB: Terrier counts documents as integers from 0 (called *docids*). It records the mapping back to *docnos* (the string form, i.e. "`d1`", "`d2`") in a separate data structure called the *metaindex*.

### Step 9: Searching an Index

Our way into search in PyTerrier is called `BatchRetrieve`. BatchRetrieve is configured by specifying an index and a weighting model (`Tf` in our example). We then search for a single-word query, `"document"`.

In [19]:
br = pt.BatchRetrieve(index, wmodel="Tf")
br.search("document")

/var/folders/ct/242_3ld56f1c5w52hw4j71w00000gn/T/ipykernel_33539/2005775709.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  br = pt.BatchRetrieve(index, wmodel="Tf")


,qid,docid,docno,rank,score,query
0,1,0,d1,0,2.0,document
1,1,1,d2,1,1.0,document
2,1,2,d3,2,1.0,document


So the `search()` method returns a dataframe with columns:
 - `qid`: this is by default "1", since it's our first and only query
 - `docid`: Terrier' internal integer for each document
 - `docno`: the external (string) unique identifier for each document
 - `score`: since we use the `Tf` weighting model, this score corresponds the total frequency of the query (terms) in each document
 - `rank`: A handy attribute showing the descending order by score
 - `query`: the input query

As expected, the `Tf` weighting model used here only counts the frequencies of the query terms in each document, i.e.:
$$
score(d,q) = \sum_{t \in q} tf_{t,d}
$$

Hence, it's clear that document `d1` should be the highest scored document with two occurrences (c.f. `'document'` and `'documents'`).  

We can also pass a dataframe of one or more queries to the `transform()` method (rather than the `search()` method) of a transformer, with queries numbered "q1", "q2" etc.. 

In [20]:
import pandas as pd
queries = pd.DataFrame([["q1", "document"], ["q2", "first document"]], columns=["qid", "query"])
br.transform(queries)

,qid,docid,docno,rank,score,query
0,q1,0,d1,0,2.0,document
1,q1,1,d2,1,1.0,document
2,q1,2,d3,2,1.0,document
3,q2,0,d1,0,3.0,first document
4,q2,1,d2,1,1.0,first document
5,q2,2,d3,2,1.0,first document


In fact, we are usually calling `transform()`, so it's the default method – i.e. 
`br.transform(queries)` can be more succinctly written as `br(queries)`.

In [21]:
br(queries)

,qid,docid,docno,rank,score,query
0,q1,0,d1,0,2.0,document
1,q1,1,d2,1,1.0,document
2,q1,2,d3,2,1.0,document
3,q2,0,d1,0,3.0,first document
4,q2,1,d2,1,1.0,first document
5,q2,2,d3,2,1.0,first document


### Step 10: scifacts corpus

OK, having 3 documents is quite trivial, so let's move upto a slightly larger corpus of documents. We'll be using the scifacts datasets for the remainder of this tutorial. PyTerrier has a handy `get_dataset()` API, which allows us to download the corpus and index it.

In [22]:
import os

# (!) irds=ir_datasets (shorthand to grab the dataset from ir_datasets source)
scifacts = pt.datasets.get_dataset('irds:beir/scifact/test')
pt_index_path = './index_scifact'

if not os.path.exists(pt_index_path + "/data.properties"):
  # create the index, using the IterDictIndexer indexer 
  indexer = pt.index.IterDictIndexer(pt_index_path, meta_reverse=[])

  # we give the dataset get_corpus_iter() directly to the indexer
  # while specifying the fields to index and the metadata to record
  index_ref = indexer.index(scifacts.get_corpus_iter())

else:
  # if you already have the index, use it.
  index_ref = pt_index_path

index = pt.IndexFactory.of(index_ref)


What are the statistics of our index?

In [23]:
#YOUR SOLUTION
print(index.getCollectionStatistics().toString())

Number of documents: 5183
Number of terms: 25244
Number of postings: 433823
Number of fields: 0
Number of tokens: 694735
Field names: []
Positions:   false



In [24]:
print(len(scifacts.get_topics()))
print(scifacts.get_topics().head(3))

300
  qid  \
0   1   
1   3   
2   5   

                                                                                                                                                 query  
0                                                                                                 0 dimensional biomaterials show inductive properties  
1  1 000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants  
2                                                                                                            1 2000 in uk have abnormal prp positivity  


Next, scifacts also has a corresponding set of queries and relevance assessments (aka qrels), thus forming a *test collection*, 

We can easily access the topics and qrels from the dataset. Indeed these are expressed as dataframes as well (we use Pandas's [`head()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.head.html) method to show only the first 5 topics):

In [25]:
scifacts.get_topics(variant='text').head(5)

,qid,query
0,1,0 dimensional biomaterials show inductive properties
1,3,1 000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants
2,5,1 2000 in uk have abnormal prp positivity
3,13,5 of perinatal mortality is due to low birth weight
4,36,a deficiency of vitamin b12 increases blood levels of homocysteine


In [26]:
scifacts.get_qrels().head(5)

,qid,docno,label,iteration
0,1,31715818,1,0
1,3,14717500,1,0
2,5,13734012,1,0
3,13,1606628,1,0
4,36,5152028,1,0


### Step 11: Weighting Models

So far, we have been using the simple "`Tf`" as our ranking function for document retrieval in BatchRetrieve. However, we can use other models such as `"TF_IDF"` by simply changing the `wmodel="Tf"` keyword argument in the constructor of `BatchRetrieve`.


In [42]:
tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")
tfidf.search("chemical reactions")

/var/folders/ct/242_3ld56f1c5w52hw4j71w00000gn/T/ipykernel_33539/2930787949.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")


,qid,docid,docno,rank,score,query
0,1,2573,15969637,0,8.490676,chemical reactions
1,1,1305,7151961,1,7.087572,chemical reactions
2,1,533,2890952,2,7.002339,chemical reactions
3,1,1144,6190603,3,6.987258,chemical reactions
4,1,5133,95764370,4,6.611113,chemical reactions
...,...,...,...,...,...,...
235,1,4065,28617573,235,1.634646,chemical reactions
236,1,1058,5691302,236,1.631678,chemical reactions
237,1,5090,76415938,237,1.577252,chemical reactions
238,1,4115,29387024,238,1.452319,chemical reactions


You will note that, as expected, the scores of documents ranked by `TF_IDF` are no longer integers. You can see the exact formula used by Terrier from [the Github repo](https://github.com/terrier-org/terrier-core/blob/5.x/modules/core/src/main/java/org/terrier/matching/models/TF_IDF.java#L79).

Terrier supports many weighting models – the documentation contains [a list of supported models](http://terrier.org/docs/current/javadoc/org/terrier/matching/models/package-summary.html) - some of which we will discover later in the tutorial.


### Step 12: Evaluating the system

So far, we have been creating search engine models, but we haven't decided if any of them are actually any good. Let's investigate if we are getting a correct ("relevant") document at the first rank.

In [43]:
qrels = scifacts.get_qrels()
def get_res_with_labels(ranker, df):
  # get the results for the query or queries
  results = ranker(df)
  # left outer join with the qrels
  with_labels = results.merge(qrels, on=["qid", "docno"], how="left").fillna(0)
  return with_labels

# lets get the Tf results for the first query
get_res_with_labels(tfidf, scifacts.get_topics(variant='text').head(1))

/var/folders/ct/242_3ld56f1c5w52hw4j71w00000gn/T/ipykernel_33539/413527676.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  with_labels = results.merge(qrels, on=["qid", "docno"], how="left").fillna(0)


,qid,docid,docno,rank,score,query,label,iteration
0,1,4399,34386619,0,8.210336,0 dimensional biomaterials show inductive properties,0.0,0
1,1,1810,10608397,1,7.993494,0 dimensional biomaterials show inductive properties,0.0,0
2,1,2891,18953920,2,7.855490,0 dimensional biomaterials show inductive properties,0.0,0
3,1,3154,21257564,3,7.777450,0 dimensional biomaterials show inductive properties,0.0,0
4,1,4721,40212412,4,7.631217,0 dimensional biomaterials show inductive properties,0.0,0
...,...,...,...,...,...,...,...,...
995,1,2908,19140422,995,2.220696,0 dimensional biomaterials show inductive properties,0.0,0
996,1,159,864491,996,2.219338,0 dimensional biomaterials show inductive properties,0.0,0
997,1,1543,8672737,997,2.219338,0 dimensional biomaterials show inductive properties,0.0,0
998,1,1557,8756719,998,2.218792,0 dimensional biomaterials show inductive properties,0.0,0


In [44]:
from pyterrier.measures import *

pt.Experiment(
    [tfidf],
    scifacts.get_topics(variant='text'),
    scifacts.get_qrels(),
    eval_metrics=["map", "ndcg"])
# we can trec_eval (string) measure names, e.g. "map", 
# or the irmeasures syntax, e.g nDCG@10

,name,map,ndcg
0,TerrierRetr(TF_IDF),0.619427,0.696696


Lets see how BM25 compares to TF_IDF - how many queries have improved (M)AP?  

In [45]:
bm25 = pt.BatchRetrieve(index, wmodel="BM25")
pt.Experiment( 
    [tfidf, bm25],
    scifacts.get_topics(variant='text'),
    scifacts.get_qrels(),
    baseline=0,
    eval_metrics=["map", "ndcg"])

/var/folders/ct/242_3ld56f1c5w52hw4j71w00000gn/T/ipykernel_33539/2460471729.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25 = pt.BatchRetrieve(index, wmodel="BM25")


,name,map,ndcg,map +,map -,map p-value,ndcg +,ndcg -,ndcg p-value
0,TerrierRetr(TF_IDF),0.619427,0.696696,NaN,NaN,NaN,NaN,NaN,NaN
1,TerrierRetr(BM25),0.626997,0.702736,52.0,29.0,0.049534,51.0,30.0,0.039061


You should see that 40 queries exhibited improvements in AP when using BM25, while 10 are degraded; This is a significant increase ($p < 0.05$). 

### Step 13 trec_eval

`trec_eval` is the standard tool in IR for evaluating ad hoc search.

Before proceeding, download and install it from here: https://github.com/usnistgov/trec_eval

If you absolutely cannot get trec_eval installed, you can also try this python alternative: https://github.com/cvangysel/pytrec_eval (see https://github.com/cvangysel/pytrec_eval/blob/master/examples/trec_eval.py for example usage).

If you have it correctly installed, the below command should work. It produces some hopefully useful information that you should read carefully.

In [31]:
!trec_eval -h

trec_eval [-h] [-q] [-m measure[.params] [-c] [-n] [-l <num>]
   [-D debug_level] [-N <num>] [-M <num>] [-R rel_format] [-T results_format]
   rel_info_file  results_file 
 
Calculate and print various evaluation measures, evaluating the results  
in results_file against the relevance info in rel_info_file. 
 
There are a fair number of options, of which only the lower case options are 
normally ever used.   
 --help:
 -h: Print full help message and exit. Full help message will include
     descriptions for any measures designated by a '-m' parameter, and
     input file format descriptions for any rel_info_format given by '-R'
     and any top results_format given by '-T.'
     Thus to see all info about preference measures use
          trec_eval -h -m all_prefs -R prefs -T trec_results 
 --version:
 -v: Print version of trec_eval and exit.
 --query_eval_wanted:
 -q: In addition to summary evaluation, give evaluation for each query/topic
 --measure measure_name[.measure_params]:
 -m

Once you have it installed, produce a trec run file of the tf-idf and BM25 experiments from above. The format of a trec run file is as follows:

qid Q0 docno rank score tag

where:

   - qid	is the query number
   - Q0	is the literal Q0
   - docno	is the id of a document returned for qid
   - rank	(1-999) is the rank of this response for this qid
   - score	is a system-dependent indication of the quality of the response
   - tag	is the identifier for the system

Example:
```
1 Q0 nhslo3844_12_012186 1 1.73315273652 mySystem
1 Q0 nhslo1393_12_003292 2 1.72581054377 mySystem
1 Q0 nhslo3844_12_002212 3 1.72522727817 mySystem
1 Q0 nhslo3844_12_012182 4 1.72522727817 mySystem
1 Q0 nhslo1393_12_003296 5 1.71374426875 mySystem
```

In [46]:
# To get your started:
run = bm25.transform(scifacts.get_topics(variant='text'))

# here
run_file = "bm25_scifact_results.txt"
pt.io.write_results(run, run_file)

tdidf = tfidf.transform(scifacts.get_topics(variant='text'))
run_tfidf_file = "tfidf_scifact_results.txt"
pt.io.write_results(tdidf, run_tfidf_file)


We can export the qrels file for our scifacts data using the ir_datasets command that was installed when we installed the library.

In [47]:
!ir_datasets export beir/scifact/test qrels > scifacts.qrels

Now that we have the run file and the qrels file, we can run trec_eval to perform our evaluation again. Make sure you replace the YOUR_RUN_FILE_NAME with the filename of the run you want to evaluate.

In [48]:
!trec_eval -c scifacts.qrels bm25_scifact_results.txt

runid                 	all	pyterrier
num_q                 	all	300
num_ret               	all	285737
num_rel               	all	339
num_rel_ret           	all	330
map                   	all	0.6270
gm_map                	all	0.2758
Rprec                 	all	0.5284
bpref                 	all	0.9700
recip_rank            	all	0.6368
iprec_at_recall_0.00  	all	0.6393
iprec_at_recall_0.10  	all	0.6393
iprec_at_recall_0.20  	all	0.6393
iprec_at_recall_0.30  	all	0.6368
iprec_at_recall_0.40  	all	0.6368
iprec_at_recall_0.50  	all	0.6333
iprec_at_recall_0.60  	all	0.6333
iprec_at_recall_0.70  	all	0.6316
iprec_at_recall_0.80  	all	0.6237
iprec_at_recall_0.90  	all	0.6157
iprec_at_recall_1.00  	all	0.6157
P_5                   	all	0.1580
P_10                  	all	0.0903
P_15                  	all	0.0620
P_20                  	all	0.0478
P_30                  	all	0.0327
P_100                 	all	0.0105
P_200                 	all	0.0053
P_500                 	all	0.0022
P_1000              

In [50]:
!trec_eval -c scifacts.qrels tfidf_scifact_results.txt

runid                 	all	pyterrier
num_q                 	all	300
num_ret               	all	285737
num_rel               	all	339
num_rel_ret           	all	330
map                   	all	0.6194
gm_map                	all	0.2680
Rprec                 	all	0.5140
bpref                 	all	0.9700
recip_rank            	all	0.6303
iprec_at_recall_0.00  	all	0.6325
iprec_at_recall_0.10  	all	0.6325
iprec_at_recall_0.20  	all	0.6325
iprec_at_recall_0.30  	all	0.6299
iprec_at_recall_0.40  	all	0.6299
iprec_at_recall_0.50  	all	0.6253
iprec_at_recall_0.60  	all	0.6253
iprec_at_recall_0.70  	all	0.6234
iprec_at_recall_0.80  	all	0.6148
iprec_at_recall_0.90  	all	0.6080
iprec_at_recall_1.00  	all	0.6080
P_5                   	all	0.1573
P_10                  	all	0.0897
P_15                  	all	0.0616
P_20                  	all	0.0477
P_30                  	all	0.0324
P_100                 	all	0.0104
P_200                 	all	0.0053
P_500                 	all	0.0022
P_1000              

Why would you want to use run files instead of using pyterrier to evaluate?

The main reason is that it gives you the most flexibility for evaluation. At any point in time you can re-compute evaluation measures without needing to re-run the experiment. Especially for very computationally intensive experiments, the run file is also beneficial in that you can save the results of your experiment as you get them without needing to wait for the enture experiment to finish (for example if your code crashes, you don't need to re-run the topics you've already got results for). 

Run file also mean you don't need to rely on PyTerrier for your experiments, if you'd like to do something more exotic or not supported in that library.

We will be using these run files extensively later in the course, so make sure you can output them.